## Leetcode 601. Human Traffic of Stadium

In [0]:
spark

In [0]:
from datetime import date
data = [(1,date(2017,1,1),10),(2,date(2017,1,2),109),(3,date(2017,1,3),150),
        (4,date(2017,1,4),99),(5,date(2017,1,5),145),(6,date(2017,1,6),1455),
        (7,date(2017,1,7),199),(8,date(2017,1,9),188)]
Stadium = spark.createDataFrame(data, ['id','visit_date','people'])
Stadium.createOrReplaceTempView('Stadium')
display(Stadium)

### Spark SQL

In [0]:
spark.sql('''
with lagged as
(
    select *,
    lag(id) over(order by id) as prev_id
    from Stadium
    where people >=100
)
,grouped as(
    select *,
    sum(case when prev_id is null or id <> prev_id + 1 then 1 else 0 end ) over(order by id) as grp
    from lagged
    
)


select id, visit_date, people
from (
    select *, count(id) over(partition by grp) as streak_len from grouped
) t
where streak_len >= 3          ''').show()

### Spark DataFrame API

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
Stadium_filtered= Stadium.filter(col('people')>=100)
win= Window.orderBy(col('visit_date'))
Stadium_grouped= Stadium_filtered.withColumn('row_num',row_number().over(win))\
    .withColumn('grp', col('id')-col('row_num'))

win1 = Window.partitionBy('grp')
Stadium_grouped.withColumn('cons_count', count('id').over(win1)).filter(col('cons_count') >=3).select('id', 'visit_date', "people").show()